# Three-way chatbot conversation, one model each via OpenRouter

Alex, Blake and Mike take turns in a single group chat. Each of them is powered by a
different frontier model, all reached through one OpenRouter client - only the model
slug differs.

The multi-agent trick here is that every agent gets **one system prompt and one user
prompt**, where the user prompt carries the whole transcript so far. That scales to any
number of participants without juggling `assistant`/`user` roles per speaker.

Requires `OPENROUTER_API_KEY` in your `.env`.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

openrouter = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

In [ ]:
class Message:
    def __init__(self, agent: "ChatBot", content: str):
        self.agent = agent
        self.content = content

    def __str__(self):
        return f"{self.agent.name}: {self.content}"

class Chat:
    def __init__(self):
        self.agents = []
        self.messages = []

    def join(self, agent: "ChatBot"):
        self.agents.append(agent)
        agent.chat = self

    def add_message(self, agent: "ChatBot", content: str):
        self.messages.append(Message(agent, content))

    @property
    def transcript(self) -> str:
        return "\n".join(str(message) for message in self.messages)


class ChatBot:
    def __init__(self, model, name: str, personality: str):
        self.model = model
        self.name = name
        self.personality = personality
        self.chat = None

    @property
    def peers(self) -> list["ChatBot"]:
        if self.chat is None:
            return []
        return [other for other in self.chat.agents if other is not self]

    @property
    def in_conversation(self) -> bool:
        return bool(self.peers)

    @property
    def system_prompt(self) -> str:
        prompt = f"Your name is {self.name}. Your personality can be described in the following way: {self.personality}."
        if self.peers:
            names = ", ".join(peer.name for peer in self.peers)
            prompt += f" You are now joined in conversation with the following people: {names}."
        return prompt

    def join_chat(self, chat: "Chat"):
        chat.join(self)

    def write_message(self, content: str):
        if self.chat is None:
            raise RuntimeError(f"{self.name} has not joined a chat")
        self.chat.add_message(self, content)

    @property
    def user_prompt(self) -> str:
        peer_names = ", ".join(peer.name for peer in self.peers)
        history = f"The conversation so far is as follows:\n{self.chat.transcript}" if self.chat.transcript else "The conversation has not started yet - you go first."
        return (
            f"You are {self.name}, in conversation with {peer_names}.\n"
            f"{history}\n"
            f"Now with this, respond with what you would like to say next, as {self.name}. "
            f"Reply with the content of your message only, without your name as a prefix."
        )

    def respond(self) -> str:
        if self.chat is None:
            raise RuntimeError(f"{self.name} has not joined a chat")

        response = openrouter.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": self.user_prompt},
            ],
        )
        content = response.choices[0].message.content
        self.write_message(content)
        return content

In [ ]:
def main():
    chat = Chat()

    alex = ChatBot("openai/gpt-4.1-mini", "Alex", "Very argumentative; you disagree with anything in the conversation and you challenge everything, in a snarky way.")
    blake = ChatBot("anthropic/claude-haiku-4.5", "Blake", "Very polite, courteous chatbot. You try to agree with everything the other person says, or find common ground. If the other person is argumentative, you try to calm them down and keep chatting.")
    mike = ChatBot("google/gemini-2.5-flash", "Mike", "Dry and matter-of-fact. You have no patience for drama and keep dragging the conversation back to concrete facts, often with a deadpan one-liner.")

    alex.join_chat(chat)
    blake.join_chat(chat)
    mike.join_chat(chat)

    for turn in range(6):
        agent = chat.agents[turn % len(chat.agents)]
        agent.respond()
        print(chat.messages[-1])


main()

## Things worth playing with

- Add a fourth agent - `peers` is computed from the chat, so everyone finds out about
  each other with no extra wiring.
- Swap the models. Current slugs are listed at https://openrouter.ai/models
- Give an agent an Ollama model instead, by handing it its own client.